In [2]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("payment_anomaly_features.csv", low_memory=False)

print("Shape:", df.shape)

print("\n--- DATA TYPES ---")
print(df.dtypes)

print("\n--- FINANCIAL YEAR ---")
print(df["financial_year"].value_counts(dropna=False).sort_index())

print("\n--- PAYMENT STATUS ---")
print(df["payment_status"].value_counts(dropna=False))

print("\n--- PAYMENT SEQUENCE ---")
print(df["payment_sequence_number"].describe())

print("\n--- PAYMENT AMOUNT ---")
print(df["fund_disbursed_amount"].describe())

print("\n--- SANCTION AMOUNT ---")
print(df["sanction_amount"].describe())

print("\n--- UNIQUE WORKS ---")
print(df["work_id"].nunique())

print("\n--- UNIQUE PAYMENTS ---")
print(df["payment_id"].nunique())

print("\n--- DUPLICATE ROWS ---")
print(df.duplicated().sum())

print("\n--- MISSING VALUES ---")
print(df.isna().sum().sort_values(ascending=False))

Shape: (109234, 32)

--- DATA TYPES ---
payment_id                                     str
house                                          str
work_id                                        str
mp_key                                         str
state                                          str
ida                                            str
constituency                                   str
elected_nominated                              str
vendor_name_raw                                str
vendor_name_normalized                         str
work_category                                  str
financial_year                                 str
payment_status                                 str
is_success                                   int64
is_in_progress                               int64
fund_disbursed_amount                      float64
expenditure_date                               str
sanction_date                                  str
sanction_amount                           

In [3]:
# Numerical feature audit

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Number of numerical columns:", len(numeric_cols))
print("\nNumerical columns:")
for col in numeric_cols:
    print("-", col)

print("\n" + "="*80)
print("DESCRIPTIVE STATISTICS")
print("="*80)

display(
    df[numeric_cols]
    .describe()
    .T
    .round(4)
)

print("\n" + "="*80)
print("NUMBER OF UNIQUE VALUES")
print("="*80)

for col in numeric_cols:
    print(f"{col:45s} : {df[col].nunique():>8}")

print("\n" + "="*80)
print("CORRELATION WITH PAYMENT AMOUNT")
print("="*80)

corr_with_payment = (
    df[numeric_cols]
    .corr(numeric_only=True)["fund_disbursed_amount"]
    .sort_values(ascending=False)
)

display(corr_with_payment.to_frame("correlation"))

Number of numerical columns: 17

Numerical columns:
- is_success
- is_in_progress
- fund_disbursed_amount
- sanction_amount
- payment_sequence_number
- days_since_previous_payment
- days_since_sanction
- prior_payment_count
- prior_disbursed_amount
- prior_vendor_count
- payment_share_of_prior_disbursed_amount
- payment_share_of_sanction_amount
- cum_disbursed_amount_at_payment
- cum_disbursed_to_sanction_ratio
- vendor_previous_payment_count
- vendor_previous_work_count
- vendor_previous_total_amount

DESCRIPTIVE STATISTICS


,count,mean,std,min,25%,50%,75%,max
is_success,109234.0,9.721000e-01,1.648000e-01,0.00,1.0000,1.0000,1.000000e+00,1.0
is_in_progress,109234.0,2.790000e-02,1.648000e-01,0.00,0.0000,0.0000,0.000000e+00,1.0
fund_disbursed_amount,109234.0,3.678633e+05,6.510190e+05,0.01,70000.0000,199955.0000,4.553255e+05,32562250.0
sanction_amount,109234.0,8.321548e+05,2.762263e+06,8448.00,200000.0000,445200.0000,8.000000e+05,73500000.0
payment_sequence_number,109234.0,2.536900e+00,5.955100e+00,1.00,1.0000,1.0000,2.000000e+00,191.0
days_since_previous_payment,37347.0,6.878090e+01,1.004534e+02,0.00,0.0000,10.0000,1.130000e+02,933.0
days_since_sanction,109234.0,1.336114e+02,1.224983e+02,0.00,32.0000,105.0000,2.020000e+02,976.0
prior_payment_count,109234.0,1.536900e+00,5.955100e+00,0.00,0.0000,0.0000,1.000000e+00,190.0
prior_disbursed_amount,109234.0,2.502998e+05,1.742866e+06,0.00,0.0000,0.0000,1.339300e+05,71238429.0
prior_vendor_count,109234.0,9.706000e-01,2.791900e+00,0.00,0.0000,0.0000,1.000000e+00,44.0



NUMBER OF UNIQUE VALUES
is_success                                    :        2
is_in_progress                                :        2
fund_disbursed_amount                         :    39865
sanction_amount                               :    16446
payment_sequence_number                       :      191
days_since_previous_payment                   :      562
days_since_sanction                           :      732
prior_payment_count                           :      191
prior_disbursed_amount                        :    23183
prior_vendor_count                            :       45
payment_share_of_prior_disbursed_amount       :    25243
payment_share_of_sanction_amount              :    34458
cum_disbursed_amount_at_payment               :    48170
cum_disbursed_to_sanction_ratio               :    36973
vendor_previous_payment_count                 :     1271
vendor_previous_work_count                    :      785
vendor_previous_total_amount                  :    67385

CORRE

,correlation
fund_disbursed_amount,1.000000
cum_disbursed_amount_at_payment,0.456077
sanction_amount,0.451677
payment_share_of_sanction_amount,0.235723
prior_disbursed_amount,0.134304
vendor_previous_total_amount,0.098617
cum_disbursed_to_sanction_ratio,0.083493
days_since_previous_payment,0.054741
is_success,0.013356
payment_share_of_prior_disbursed_amount,0.007650


In [4]:
print("=" * 80)
print("MATHEMATICAL REDUNDANCY AUDIT")
print("=" * 80)

# 1. Payment sequence vs prior payment count
seq_check = (
    df["payment_sequence_number"] - 1
    == df["prior_payment_count"]
)

print("\n1. payment_sequence_number - 1 == prior_payment_count")
print("Matching rows:", seq_check.sum())
print("Total rows:", len(df))
print("Match rate:", round(seq_check.mean() * 100, 4), "%")


# 2. Success vs in-progress
status_check = (
    df["is_success"] + df["is_in_progress"] == 1
)

print("\n2. is_success + is_in_progress == 1")
print("Matching rows:", status_check.sum())
print("Total rows:", len(df))
print("Match rate:", round(status_check.mean() * 100, 4), "%")


# 3. Cumulative amount vs prior amount + current payment
cum_amount_check = (
    df["cum_disbursed_amount_at_payment"]
    - (
        df["prior_disbursed_amount"]
        + df["fund_disbursed_amount"]
    )
).abs() < 0.01

print("\n3. cumulative amount == prior amount + current payment")
print("Matching rows:", cum_amount_check.sum())
print("Total rows:", len(df))
print("Match rate:", round(cum_amount_check.mean() * 100, 4), "%")


# 4. Cumulative ratio vs cumulative amount / sanction
cum_ratio_check = (
    df["cum_disbursed_to_sanction_ratio"]
    - (
        df["cum_disbursed_amount_at_payment"]
        / df["sanction_amount"]
    )
).abs() < 1e-6

print("\n4. cumulative ratio == cumulative amount / sanction amount")
print("Matching rows:", cum_ratio_check.sum())
print("Total rows:", len(df))
print("Match rate:", round(cum_ratio_check.mean() * 100, 4), "%")


# 5. Payment share of sanction vs payment / sanction
payment_share_check = (
    df["payment_share_of_sanction_amount"]
    - (
        df["fund_disbursed_amount"]
        / df["sanction_amount"]
    )
).abs() < 1e-6

print("\n5. payment share of sanction == payment / sanction")
print("Matching rows:", payment_share_check.sum())
print("Total rows:", len(df))
print("Match rate:", round(payment_share_check.mean() * 100, 4), "%")

MATHEMATICAL REDUNDANCY AUDIT

1. payment_sequence_number - 1 == prior_payment_count
Matching rows: 109234
Total rows: 109234
Match rate: 100.0 %

2. is_success + is_in_progress == 1
Matching rows: 109234
Total rows: 109234
Match rate: 100.0 %

3. cumulative amount == prior amount + current payment
Matching rows: 109234
Total rows: 109234
Match rate: 100.0 %

4. cumulative ratio == cumulative amount / sanction amount
Matching rows: 109234
Total rows: 109234
Match rate: 100.0 %

5. payment share of sanction == payment / sanction
Matching rows: 109234
Total rows: 109234
Match rate: 100.0 %


In [5]:
print("=" * 80)
print("PAYMENT FEATURE TEMPORAL AUDIT")
print("=" * 80)

audit_cols = [
    "payment_id",
    "work_id",
    "fund_disbursed_amount",
    "payment_sequence_number",
    "days_since_previous_payment",
    "days_since_sanction",
    "prior_payment_count",
    "prior_disbursed_amount",
    "prior_vendor_count",
    "payment_share_of_prior_disbursed_amount",
    "payment_share_of_sanction_amount",
    "cum_disbursed_amount_at_payment",
    "cum_disbursed_to_sanction_ratio",
    "vendor_previous_payment_count",
    "vendor_previous_work_count",
    "vendor_previous_total_amount"
]

display(df[audit_cols].head(10))

PAYMENT FEATURE TEMPORAL AUDIT


,payment_id,work_id,fund_disbursed_amount,payment_sequence_number,days_since_previous_payment,days_since_sanction,prior_payment_count,prior_disbursed_amount,prior_vendor_count,payment_share_of_prior_disbursed_amount,payment_share_of_sanction_amount,cum_disbursed_amount_at_payment,cum_disbursed_to_sanction_ratio,vendor_previous_payment_count,vendor_previous_work_count,vendor_previous_total_amount
0,PAY_LS_000001,WS/MP18218/2025-2026/233777,799146.0,1,NaN,252,0,0.0,0,NaN,0.998933,799146.0,0.998933,3,3,2328152.0
1,PAY_LS_000002,WS/MP18218/2025-2026/233878,832080.0,1,NaN,235,0,0.0,0,NaN,0.990571,832080.0,0.990571,1,1,997763.0
2,PAY_LS_000003,WS/MP18218/2025-2026/233781,498309.0,1,NaN,235,0,0.0,0,NaN,0.996618,498309.0,0.996618,2,2,1829843.0
3,PAY_LS_000004,WS/MP370/2024-2025/145537,2000.0,8,102.0,628,7,480880.0,7,0.004159,0.002857,482880.0,0.689829,435,424,1004862.0
4,PAY_LS_000005,WS/MP18350/2025-2026/182265,3000.0,1,NaN,284,0,0.0,0,NaN,0.010000,3000.0,0.010000,418,408,948062.0
5,PAY_LS_000006,WS/MP18082/2025-2026/187502,250000.0,2,285.0,305,1,750000.0,1,0.333333,0.250000,1000000.0,1.000000,342,287,244952562.0
6,PAY_LS_000007,WS/MP18082/2025-2026/242466,375000.0,1,NaN,158,0,0.0,0,NaN,0.750000,375000.0,0.750000,354,290,248514848.0
7,PAY_LS_000008,WS/MP641/2025-2026/220322,99254.0,2,161.0,364,1,100000.0,1,0.992540,0.496270,199254.0,0.996270,11,7,3299999.0
8,PAY_LS_000009,WS/MP337/2025-2026/247818,124750.0,2,221.0,225,1,374250.0,1,0.333333,0.250000,499000.0,1.000000,357,293,250764848.0
9,PAY_LS_000010,WS/MP18222/2025-2026/184339,19992.0,1,NaN,407,0,0.0,0,NaN,0.956555,19992.0,0.956555,241,236,38959660.0


In [6]:
print("=" * 80)
print("TEMPORAL CONSISTENCY CHECK")
print("=" * 80)

# Convert dates
df["expenditure_date"] = pd.to_datetime(
    df["expenditure_date"],
    errors="coerce"
)

df["sanction_date"] = pd.to_datetime(
    df["sanction_date"],
    errors="coerce"
)

print("\nMissing dates:")
print("expenditure_date:", df["expenditure_date"].isna().sum())
print("sanction_date:", df["sanction_date"].isna().sum())


# --------------------------------------------------
# 1. Check payment sequence vs expenditure date
# --------------------------------------------------

work_order_check = (
    df.sort_values(["work_id", "payment_sequence_number"])
      .groupby("work_id")["expenditure_date"]
      .apply(lambda x: x.is_monotonic_increasing)
)

print("\n" + "=" * 80)
print("1. PAYMENT SEQUENCE vs EXPENDITURE DATE")
print("=" * 80)

print("Works with chronological payment order:",
      work_order_check.sum())

print("Total works:", len(work_order_check))

print("Chronological order rate:",
      round(work_order_check.mean() * 100, 4), "%")


# --------------------------------------------------
# 2. Check days_since_previous_payment
# --------------------------------------------------

temp = df.sort_values(
    ["work_id", "payment_sequence_number"]
).copy()

temp["calculated_days_since_previous"] = (
    temp.groupby("work_id")["expenditure_date"]
        .diff()
        .dt.days
)

comparison = temp[
    temp["days_since_previous_payment"].notna()
    & temp["calculated_days_since_previous"].notna()
].copy()

comparison["difference"] = (
    comparison["days_since_previous_payment"]
    - comparison["calculated_days_since_previous"]
)

print("\n" + "=" * 80)
print("2. DAYS SINCE PREVIOUS PAYMENT")
print("=" * 80)

print("Rows compared:", len(comparison))

print("Exact matches:",
      (comparison["difference"] == 0).sum())

print("Exact match rate:",
      round(
          (comparison["difference"] == 0).mean() * 100,
          4
      ),
      "%")

print("\nDifference summary:")
display(
    comparison["difference"]
    .describe()
    .to_frame("difference")
    .round(4)
)


# --------------------------------------------------
# 3. Check days_since_sanction
# --------------------------------------------------

sanction_comparison = df[
    df["days_since_sanction"].notna()
    & df["expenditure_date"].notna()
    & df["sanction_date"].notna()
].copy()

sanction_comparison["calculated_days_since_sanction"] = (
    sanction_comparison["expenditure_date"]
    - sanction_comparison["sanction_date"]
).dt.days

sanction_comparison["difference"] = (
    sanction_comparison["days_since_sanction"]
    - sanction_comparison["calculated_days_since_sanction"]
)

print("\n" + "=" * 80)
print("3. DAYS SINCE SANCTION")
print("=" * 80)

print("Rows compared:", len(sanction_comparison))

print("Exact matches:",
      (sanction_comparison["difference"] == 0).sum())

print("Exact match rate:",
      round(
          (sanction_comparison["difference"] == 0).mean() * 100,
          4
      ),
      "%")

print("\nDifference summary:")
display(
    sanction_comparison["difference"]
    .describe()
    .to_frame("difference")
    .round(4)
)

TEMPORAL CONSISTENCY CHECK

Missing dates:
expenditure_date: 0
sanction_date: 0

1. PAYMENT SEQUENCE vs EXPENDITURE DATE
Works with chronological payment order: 71887
Total works: 71887
Chronological order rate: 100.0 %

2. DAYS SINCE PREVIOUS PAYMENT
Rows compared: 37347
Exact matches: 37347
Exact match rate: 100.0 %

Difference summary:


,difference
count,37347.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0



3. DAYS SINCE SANCTION
Rows compared: 109234
Exact matches: 109234
Exact match rate: 100.0 %

Difference summary:


,difference
count,109234.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0


In [7]:
print("=" * 80)
print("VENDOR HISTORY LEAKAGE CHECK")
print("=" * 80)

vendor_cols = [
    "vendor_previous_payment_count",
    "vendor_previous_work_count",
    "vendor_previous_total_amount"
]

print("\nMissing values:")
display(df[vendor_cols].isna().sum().to_frame("missing_count"))

print("\nBasic statistics:")
display(
    df[vendor_cols]
    .describe()
    .T
    .round(4)
)

print("\nFirst 10 rows:")
display(
    df[
        [
            "payment_id",
            "vendor_name_normalized",
            "expenditure_date",
            "fund_disbursed_amount",
            "vendor_previous_payment_count",
            "vendor_previous_work_count",
            "vendor_previous_total_amount"
        ]
    ].head(10)
)

VENDOR HISTORY LEAKAGE CHECK

Missing values:


,missing_count
vendor_previous_payment_count,0
vendor_previous_work_count,0
vendor_previous_total_amount,0



Basic statistics:


,count,mean,std,min,25%,50%,75%,max
vendor_previous_payment_count,109234.0,3.911610e+01,1.015150e+02,0.0,0.0,4.0,32.00,1270.0
vendor_previous_work_count,109234.0,3.127320e+01,7.686860e+01,0.0,0.0,3.0,27.00,784.0
vendor_previous_total_amount,109234.0,9.447597e+06,2.218639e+07,0.0,0.0,990380.5,8082270.75,253139348.0



First 10 rows:


,payment_id,vendor_name_normalized,expenditure_date,fund_disbursed_amount,vendor_previous_payment_count,vendor_previous_work_count,vendor_previous_total_amount
0,PAY_LS_000001,DARSH BUILDCON,2026-08-21,799146.0,3,3,2328152.0
1,PAY_LS_000002,DARSH BUILDCON,2026-08-04,832080.0,1,1,997763.0
2,PAY_LS_000003,DARSH BUILDCON,2026-08-04,498309.0,2,2,1829843.0
3,PAY_LS_000004,MEMBER SECY OB AND OC WWB BBSR,2026-08-31,2000.0,435,424,1004862.0
4,PAY_LS_000005,MEMBER SECY OB AND OC WWB BBSR,2026-08-17,3000.0,418,408,948062.0
5,PAY_LS_000006,KRIDL BHUSIRI ACCOUNT WORKS,2026-08-10,250000.0,342,287,244952562.0
6,PAY_LS_000007,KRIDL BHUSIRI ACCOUNT WORKS,2026-08-31,375000.0,354,290,248514848.0
7,PAY_LS_000008,NAGAR PALIKA PARISHAD BARWANI,2026-08-31,99254.0,11,7,3299999.0
8,PAY_LS_000009,KRIDL BHUSIRI ACCOUNT WORKS,2026-09-05,124750.0,357,293,250764848.0
9,PAY_LS_000010,AJAY KUMAR SINGH,2026-07-22,19992.0,241,236,38959660.0


In [9]:
print("=" * 80)
print("VENDOR HISTORY TEMPORAL CONSISTENCY TEST")
print("=" * 80)

# Sort payments by vendor, date, and payment ID
vendor_test = df.sort_values(
    ["vendor_name_normalized", "expenditure_date", "payment_id"]
).copy()

# --------------------------------------------------
# Reconstruct previous vendor payment count
# --------------------------------------------------

vendor_test["calc_vendor_previous_payment_count"] = (
    vendor_test.groupby("vendor_name_normalized")
    .cumcount()
)

# --------------------------------------------------
# Reconstruct previous vendor total amount
# --------------------------------------------------

vendor_test["calc_vendor_previous_total_amount"] = (
    vendor_test.groupby("vendor_name_normalized")[
        "fund_disbursed_amount"
    ].cumsum()
    - vendor_test["fund_disbursed_amount"]
)

# --------------------------------------------------
# Reconstruct previous distinct work count
# --------------------------------------------------

seen_works = {}
previous_work_counts = []

for vendor, work_id in zip(
    vendor_test["vendor_name_normalized"],
    vendor_test["work_id"]
):
    if vendor not in seen_works:
        seen_works[vendor] = set()

    previous_work_counts.append(
        len(seen_works[vendor])
    )

    seen_works[vendor].add(work_id)

vendor_test["calc_vendor_previous_work_count"] = (
    previous_work_counts
)

# --------------------------------------------------
# Compare stored vs reconstructed values
# --------------------------------------------------

vendor_test["payment_count_match"] = (
    vendor_test["vendor_previous_payment_count"]
    == vendor_test["calc_vendor_previous_payment_count"]
)

vendor_test["work_count_match"] = (
    vendor_test["vendor_previous_work_count"]
    == vendor_test["calc_vendor_previous_work_count"]
)

vendor_test["amount_match"] = (
    (
        vendor_test["vendor_previous_total_amount"]
        - vendor_test["calc_vendor_previous_total_amount"]
    ).abs() < 0.01
)

# --------------------------------------------------
# Results
# --------------------------------------------------

print("\nVendor previous payment count:")
print(
    "Match rate:",
    round(
        vendor_test["payment_count_match"].mean() * 100,
        4
    ),
    "%"
)

print("\nVendor previous work count:")
print(
    "Match rate:",
    round(
        vendor_test["work_count_match"].mean() * 100,
        4
    ),
    "%"
)

print("\nVendor previous total amount:")
print(
    "Match rate:",
    round(
        vendor_test["amount_match"].mean() * 100,
        4
    ),
    "%"
)

print("\n" + "=" * 80)
print("MISMATCH COUNTS")
print("=" * 80)

print(
    "Payment count mismatches:",
    (~vendor_test["payment_count_match"]).sum()
)

print(
    "Work count mismatches:",
    (~vendor_test["work_count_match"]).sum()
)

print(
    "Amount mismatches:",
    (~vendor_test["amount_match"]).sum()
)

VENDOR HISTORY TEMPORAL CONSISTENCY TEST

Vendor previous payment count:
Match rate: 100.0 %

Vendor previous work count:
Match rate: 100.0 %

Vendor previous total amount:
Match rate: 100.0 %

MISMATCH COUNTS
Payment count mismatches: 0
Work count mismatches: 0
Amount mismatches: 0


In [10]:
print("=" * 80)
print("STRUCTURAL MISSINGNESS CHECK")
print("=" * 80)

first_payment = df["payment_sequence_number"] == 1

for col in [
    "days_since_previous_payment",
    "payment_share_of_prior_disbursed_amount"
]:
    missing = df[col].isna()

    print(f"\n{col}")
    print("Missing values:", missing.sum())
    print("First payments:", first_payment.sum())
    print("Missing AND first payment:", (missing & first_payment).sum())
    print("Missing but NOT first payment:", (missing & ~first_payment).sum())

STRUCTURAL MISSINGNESS CHECK

days_since_previous_payment
Missing values: 71887
First payments: 71887
Missing AND first payment: 71887
Missing but NOT first payment: 0

payment_share_of_prior_disbursed_amount
Missing values: 71887
First payments: 71887
Missing AND first payment: 71887
Missing but NOT first payment: 0


In [11]:
candidate_features = [
    "fund_disbursed_amount",
    "sanction_amount",
    "payment_sequence_number",
    "days_since_previous_payment",
    "days_since_sanction",
    "prior_disbursed_amount",
    "prior_vendor_count",
    "payment_share_of_prior_disbursed_amount",
    "vendor_previous_payment_count",
    "vendor_previous_work_count",
    "vendor_previous_total_amount",
    "is_success"
]

print("Number of candidate features:", len(candidate_features))
print("\nCandidate features:")
for i, col in enumerate(candidate_features, 1):
    print(f"{i:2}. {col}")

Number of candidate features: 12

Candidate features:
 1. fund_disbursed_amount
 2. sanction_amount
 3. payment_sequence_number
 4. days_since_previous_payment
 5. days_since_sanction
 6. prior_disbursed_amount
 7. prior_vendor_count
 8. payment_share_of_prior_disbursed_amount
 9. vendor_previous_payment_count
10. vendor_previous_work_count
11. vendor_previous_total_amount
12. is_success


In [12]:
print("=" * 80)
print("TEMPORAL TRAIN / TEST SPLIT")
print("=" * 80)

train_years = [
    "2023-2024",
    "2024-2025",
    "2025-2026"
]

test_years = [
    "2026-2027"
]

train_df = df[df["financial_year"].isin(train_years)].copy()
test_df = df[df["financial_year"].isin(test_years)].copy()

print("\nTRAIN")
print("Rows:", len(train_df))
print("Financial years:")
print(train_df["financial_year"].value_counts().sort_index())

print("\nTEST")
print("Rows:", len(test_df))
print("Financial years:")
print(test_df["financial_year"].value_counts().sort_index())

print("\n" + "=" * 80)
print("FEATURE AVAILABILITY")
print("=" * 80)

print("\nTraining missing values:")
display(train_df[candidate_features].isna().sum().to_frame("missing"))

print("\nTesting missing values:")
display(test_df[candidate_features].isna().sum().to_frame("missing"))

TEMPORAL TRAIN / TEST SPLIT

TRAIN
Rows: 100603
Financial years:
financial_year
2023-2024     5185
2024-2025    26558
2025-2026    68860
Name: count, dtype: int64

TEST
Rows: 8631
Financial years:
financial_year
2026-2027    8631
Name: count, dtype: int64

FEATURE AVAILABILITY

Training missing values:


,missing
fund_disbursed_amount,0
sanction_amount,0
payment_sequence_number,0
days_since_previous_payment,64915
days_since_sanction,0
prior_disbursed_amount,0
prior_vendor_count,0
payment_share_of_prior_disbursed_amount,64915
vendor_previous_payment_count,0
vendor_previous_work_count,0



Testing missing values:


,missing
fund_disbursed_amount,0
sanction_amount,0
payment_sequence_number,0
days_since_previous_payment,6972
days_since_sanction,0
prior_disbursed_amount,0
prior_vendor_count,0
payment_share_of_prior_disbursed_amount,6972
vendor_previous_payment_count,0
vendor_previous_work_count,0


In [13]:
from sklearn.impute import SimpleImputer
from sklearn.ensemble import IsolationForest

# Prepare feature matrices
X_train = train_df[candidate_features].copy()
X_test = test_df[candidate_features].copy()

# Structural missing values:
# -1 means "no previous payment exists"
imputer = SimpleImputer(
    strategy="constant",
    fill_value=-1
)

X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

print("Training matrix shape:", X_train_imp.shape)
print("Testing matrix shape:", X_test_imp.shape)

# --------------------------------------------------
# Baseline Isolation Forest
# --------------------------------------------------

payment_model = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

payment_model.fit(X_train_imp)

# Isolation Forest: lower score = more anomalous
train_scores = -payment_model.score_samples(X_train_imp)
test_scores = -payment_model.score_samples(X_test_imp)

print("\n" + "=" * 80)
print("BASELINE MODEL TRAINED")
print("=" * 80)

print("\nTraining anomaly scores:")
print(pd.Series(train_scores).describe().round(4))

print("\nTesting anomaly scores:")
print(pd.Series(test_scores).describe().round(4))

Training matrix shape: (100603, 12)
Testing matrix shape: (8631, 12)

BASELINE MODEL TRAINED

Training anomaly scores:
count    100603.0000
mean          0.3781
std           0.0651
min           0.3207
25%           0.3319
50%           0.3537
75%           0.3993
max           0.7645
dtype: float64

Testing anomaly scores:
count    8631.0000
mean        0.3843
std         0.0710
min         0.3214
25%         0.3318
50%         0.3521
75%         0.4217
max         0.7593
dtype: float64


In [14]:
print("=" * 80)
print("ANOMALY SCORE DISTRIBUTION")
print("=" * 80)

percentiles = [90, 95, 97, 98, 99, 99.5, 99.9]

train_percentiles = np.percentile(
    train_scores,
    percentiles
)

test_percentiles = np.percentile(
    test_scores,
    percentiles
)

score_table = pd.DataFrame({
    "percentile": percentiles,
    "train_score": train_percentiles,
    "test_score": test_percentiles
})

display(score_table.round(6))

print("\nTraining score quantiles:")
print(
    pd.Series(train_scores)
    .quantile([0.90, 0.95, 0.97, 0.98, 0.99, 0.995, 0.999])
    .round(6)
)

print("\nMaximum training score:", round(train_scores.max(), 6))
print("Maximum testing score:", round(test_scores.max(), 6))

ANOMALY SCORE DISTRIBUTION


,percentile,train_score,test_score
0,90.0,0.470163,0.488528
1,95.0,0.516513,0.518172
2,97.0,0.548272,0.544734
3,98.0,0.575713,0.564176
4,99.0,0.619199,0.618262
5,99.5,0.660933,0.661325
6,99.9,0.706119,0.734754



Training score quantiles:
0.900    0.470163
0.950    0.516513
0.970    0.548272
0.980    0.575713
0.990    0.619199
0.995    0.660933
0.999    0.706119
dtype: float64

Maximum training score: 0.764474
Maximum testing score: 0.759301


In [15]:
print("=" * 80)
print("TRAINING-DERIVED ANOMALY THRESHOLDS")
print("=" * 80)

thresholds = {
    "p95": np.percentile(train_scores, 95),
    "p99": np.percentile(train_scores, 99),
    "p99.5": np.percentile(train_scores, 99.5),
    "p99.9": np.percentile(train_scores, 99.9)
}

rows = []

for name, threshold in thresholds.items():
    train_count = (train_scores >= threshold).sum()
    test_count = (test_scores >= threshold).sum()

    rows.append({
        "threshold": name,
        "score_cutoff": threshold,
        "train_anomalies": train_count,
        "train_rate_%": train_count / len(train_scores) * 100,
        "test_anomalies": test_count,
        "test_rate_%": test_count / len(test_scores) * 100
    })

threshold_table = pd.DataFrame(rows)

display(threshold_table.round(4))

TRAINING-DERIVED ANOMALY THRESHOLDS


,threshold,score_cutoff,train_anomalies,train_rate_%,test_anomalies,test_rate_%
0,p95,0.5165,5031,5.0008,455,5.2717
1,p99,0.6192,1007,1.0010,83,0.9616
2,p99.5,0.6609,504,0.5010,45,0.5214
3,p99.9,0.7061,101,0.1004,25,0.2897


In [16]:


print("=" * 80)
print("TOP PAYMENT ANOMALIES — UNSEEN 2026-2027")
print("=" * 80)

# Primary operational threshold: training p99
payment_threshold = thresholds["p99"]

test_results = test_df.copy()
test_results["anomaly_score"] = test_scores
test_results["payment_anomaly"] = (
    test_results["anomaly_score"] >= payment_threshold
)

top_anomalies = (
    test_results
    .sort_values("anomaly_score", ascending=False)
    .head(20)
)

display(
    top_anomalies[
        [
            "payment_id",
            "work_id",
            "vendor_name_normalized",
            "expenditure_date",
            "fund_disbursed_amount",
            "sanction_amount",
            "payment_sequence_number",
            "days_since_previous_payment",
            "days_since_sanction",
            "prior_disbursed_amount",
            "payment_share_of_prior_disbursed_amount",
            "vendor_previous_payment_count",
            "vendor_previous_work_count",
            "vendor_previous_total_amount",
            "anomaly_score",
            "payment_anomaly"
        ]
    ].round(4)
)

print("\nThreshold:", round(payment_threshold, 6))
print("Flagged test payments:", test_results["payment_anomaly"].sum())
print(
    "Flagged rate:",
    round(test_results["payment_anomaly"].mean() * 100, 4),
    "%"
)

TOP PAYMENT ANOMALIES — UNSEEN 2026-2027


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_2148\576399341.py:40: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(4)


,payment_id,work_id,vendor_name_normalized,expenditure_date,fund_disbursed_amount,sanction_amount,payment_sequence_number,days_since_previous_payment,days_since_sanction,prior_disbursed_amount,payment_share_of_prior_disbursed_amount,vendor_previous_payment_count,vendor_previous_work_count,vendor_previous_total_amount,anomaly_score,payment_anomaly
73973,PAY_LS_073974,WS/MP507/2026-2027/290535,SHRI GURU KRIPA NARSINGH ASSOCIATES,2026-08-24,5651433.0,49740000.0,8,0.0,5,29670024.0,0.1905,132,38,164515577.0,0.7593,True
73972,PAY_LS_073973,WS/MP507/2026-2027/290535,SHRI GURU KRIPA NARSINGH ASSOCIATES,2026-08-24,4238575.0,49740000.0,7,0.0,5,25431449.0,0.1667,131,38,160277002.0,0.7516,True
108659,PAY_RS_024552,WS/MP187/2026-2027/283439,HIDAYA QIRAT ENTERPRISES,2026-06-23,3494400.0,20716800.0,7,0.0,8,17222400.0,0.2029,344,276,196634150.0,0.7515,True
73971,PAY_LS_073972,WS/MP507/2026-2027/290535,SHRI GURU KRIPA NARSINGH ASSOCIATES,2026-08-24,3767622.0,49740000.0,6,0.0,5,21663827.0,0.1739,130,38,156509380.0,0.7467,True
108658,PAY_RS_024551,WS/MP187/2026-2027/283439,HIDAYA QIRAT ENTERPRISES,2026-06-23,2745600.0,20716800.0,6,0.0,8,14476800.0,0.1897,343,276,193888550.0,0.7456,True
73970,PAY_LS_073971,WS/MP507/2026-2027/290535,SHRI GURU KRIPA NARSINGH ASSOCIATES,2026-08-24,4945004.0,49740000.0,5,0.0,5,16718823.0,0.2958,129,38,151564376.0,0.7424,True
108656,PAY_RS_024549,WS/MP187/2026-2027/283439,HIDAYA QIRAT ENTERPRISES,2026-06-16,3744000.0,20716800.0,4,0.0,1,8486400.0,0.4412,341,276,187898150.0,0.7376,True
108657,PAY_RS_024550,WS/MP187/2026-2027/283439,HIDAYA QIRAT ENTERPRISES,2026-06-23,2246400.0,20716800.0,5,7.0,8,12230400.0,0.1837,342,276,191642150.0,0.7375,True
103981,PAY_RS_019874,WS/MP203/2026-2027/284964,RAMJI CONSTRUCTION,2026-08-03,4276800.0,28512000.0,7,0.0,37,24235200.0,0.1765,54,9,150989800.0,0.7352,True
73969,PAY_LS_073970,WS/MP507/2026-2027/290535,SHRI GURU KRIPA NARSINGH ASSOCIATES,2026-08-24,3532146.0,49740000.0,4,0.0,5,13186677.0,0.2679,128,38,148032230.0,0.7345,True



Threshold: 0.619199
Flagged test payments: 83
Flagged rate: 0.9616 %


In [17]:
print("=" * 80)
print("IS THE MODEL JUST DETECTING LARGE PAYMENTS?")
print("=" * 80)

# Compare flagged vs all test payments
flagged = test_results[test_results["payment_anomaly"]].copy()
normal = test_results[~test_results["payment_anomaly"]].copy()

print("\nPayment amount:")
print("Flagged median:", round(flagged["fund_disbursed_amount"].median(), 2))
print("Overall median:", round(test_results["fund_disbursed_amount"].median(), 2))

print("\nFlagged payment amount percentiles:")
print(
    flagged["fund_disbursed_amount"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
    .round(2)
)

print("\n" + "=" * 80)
print("TOP 10 ANOMALIES")
print("=" * 80)

display(
    test_results
    .sort_values("anomaly_score", ascending=False)
    [
        [
            "payment_id",
            "fund_disbursed_amount",
            "sanction_amount",
            "payment_sequence_number",
            "days_since_previous_payment",
            "days_since_sanction",
            "prior_disbursed_amount",
            "payment_share_of_prior_disbursed_amount",
            "vendor_previous_payment_count",
            "vendor_previous_work_count",
            "anomaly_score"
        ]
    ]
    .head(10)
    .round(4)
)

print("\nCorrelation: anomaly score vs payment amount:",
      round(
          test_results[
              ["anomaly_score", "fund_disbursed_amount"]
          ].corr().iloc[0, 1],
          4
      ))

IS THE MODEL JUST DETECTING LARGE PAYMENTS?

Payment amount:
Flagged median: 2496000.0
Overall median: 260852.0

Flagged payment amount percentiles:
count          83.00
mean      2654411.08
std       2246218.97
min           168.00
25%        883800.50
50%       2496000.00
75%       3784611.00
90%       4735825.60
95%       5216640.00
max      13824453.00
Name: fund_disbursed_amount, dtype: float64

TOP 10 ANOMALIES


,payment_id,fund_disbursed_amount,sanction_amount,payment_sequence_number,days_since_previous_payment,days_since_sanction,prior_disbursed_amount,payment_share_of_prior_disbursed_amount,vendor_previous_payment_count,vendor_previous_work_count,anomaly_score
73973,PAY_LS_073974,5651433.0,49740000.0,8,0.0,5,29670024.0,0.1905,132,38,0.7593
73972,PAY_LS_073973,4238575.0,49740000.0,7,0.0,5,25431449.0,0.1667,131,38,0.7516
108659,PAY_RS_024552,3494400.0,20716800.0,7,0.0,8,17222400.0,0.2029,344,276,0.7515
73971,PAY_LS_073972,3767622.0,49740000.0,6,0.0,5,21663827.0,0.1739,130,38,0.7467
108658,PAY_RS_024551,2745600.0,20716800.0,6,0.0,8,14476800.0,0.1897,343,276,0.7456
73970,PAY_LS_073971,4945004.0,49740000.0,5,0.0,5,16718823.0,0.2958,129,38,0.7424
108656,PAY_RS_024549,3744000.0,20716800.0,4,0.0,1,8486400.0,0.4412,341,276,0.7376
108657,PAY_RS_024550,2246400.0,20716800.0,5,7.0,8,12230400.0,0.1837,342,276,0.7375
103981,PAY_RS_019874,4276800.0,28512000.0,7,0.0,37,24235200.0,0.1765,54,9,0.7352
73969,PAY_LS_073970,3532146.0,49740000.0,4,0.0,5,13186677.0,0.2679,128,38,0.7345



Correlation: anomaly score vs payment amount: 0.3001


In [18]:
print("=" * 80)
print("MODEL STABILITY TEST")
print("=" * 80)

seeds = [7, 21, 42, 84, 123]

stability_results = []

for seed in seeds:
    model = IsolationForest(
        n_estimators=300,
        contamination="auto",
        random_state=seed,
        n_jobs=-1
    )

    model.fit(X_train_imp)

    scores = -model.score_samples(X_test_imp)

    anomalies = scores >= payment_threshold

    stability_results.append({
        "seed": seed,
        "anomaly_count": anomalies.sum(),
        "anomaly_rate_%": anomalies.mean() * 100
    })

stability_table = pd.DataFrame(stability_results)

display(stability_table.round(4))

MODEL STABILITY TEST


,seed,anomaly_count,anomaly_rate_%
0,7,84,0.9732
1,21,84,0.9732
2,42,83,0.9616
3,84,76,0.8805
4,123,87,1.0080


In [19]:
print("=" * 80)
print("INDIVIDUAL ANOMALY STABILITY")
print("=" * 80)

seed_flags = {}

for seed in seeds:
    model = IsolationForest(
        n_estimators=300,
        contamination="auto",
        random_state=seed,
        n_jobs=-1
    )

    model.fit(X_train_imp)

    scores = -model.score_samples(X_test_imp)
    seed_flags[seed] = set(
        test_df.index[scores >= payment_threshold]
    )

# Count how many seeds flagged each payment
all_flagged = set().union(*seed_flags.values())

flag_frequency = pd.Series(0, index=list(all_flagged))

for seed in seeds:
    flag_frequency.loc[list(seed_flags[seed])] += 1

print("\nTotal unique payments flagged by at least one seed:",
      len(flag_frequency))

print("\nFlagged in all 5 seeds:",
      (flag_frequency == 5).sum())

print("Flagged in >= 4 seeds:",
      (flag_frequency >= 4).sum())

print("Flagged in >= 3 seeds:",
      (flag_frequency >= 3).sum())

print("Flagged in only 1 seed:",
      (flag_frequency == 1).sum())

print("\nStability distribution:")
display(
    flag_frequency.value_counts()
    .sort_index()
    .rename_axis("number_of_seeds")
    .to_frame("number_of_payments")
)

INDIVIDUAL ANOMALY STABILITY

Total unique payments flagged by at least one seed: 95

Flagged in all 5 seeds: 67
Flagged in >= 4 seeds: 76
Flagged in >= 3 seeds: 88
Flagged in only 1 seed: 7

Stability distribution:


,number_of_payments
number_of_seeds,
1,7
3,12
4,9
5,67


In [20]:
print("=" * 80)
print("STABLE ANOMALIES — BEHAVIOR PROFILE")
print("=" * 80)

stable_indices = flag_frequency[flag_frequency >= 4].index

stable = test_results.loc[stable_indices].copy()

print("\nStable anomalies (>=4/5 seeds):", len(stable))

profile_cols = [
    "fund_disbursed_amount",
    "sanction_amount",
    "payment_sequence_number",
    "days_since_previous_payment",
    "days_since_sanction",
    "prior_disbursed_amount",
    "payment_share_of_prior_disbursed_amount",
    "vendor_previous_payment_count",
    "vendor_previous_work_count",
    "vendor_previous_total_amount",
    "anomaly_score"
]

profile = pd.DataFrame({
    "stable_anomaly_median": stable[profile_cols].median(),
    "all_test_median": test_results[profile_cols].median()
})

profile["ratio_stable_to_all"] = (
    profile["stable_anomaly_median"]
    / profile["all_test_median"]
)

display(profile.round(4))

STABLE ANOMALIES — BEHAVIOR PROFILE

Stable anomalies (>=4/5 seeds): 76


,stable_anomaly_median,all_test_median,ratio_stable_to_all
fund_disbursed_amount,2.745600e+06,2.608520e+05,10.5255
sanction_amount,1.525500e+07,5.000000e+05,30.5100
payment_sequence_number,3.000000e+00,1.000000e+00,3.0000
days_since_previous_payment,0.000000e+00,0.000000e+00,NaN
days_since_sanction,5.000000e+00,1.700000e+01,0.2941
prior_disbursed_amount,4.665602e+06,0.000000e+00,inf
payment_share_of_prior_disbursed_amount,4.036000e-01,2.574000e-01,1.5680
vendor_previous_payment_count,1.315000e+02,7.000000e+00,18.7857
vendor_previous_work_count,5.350000e+01,5.000000e+00,10.7000
vendor_previous_total_amount,1.442755e+08,1.634460e+06,88.2710


In [21]:
import pickle
import json

print("=" * 80)
print("EXPORTING PAYMENT ANOMALY MODEL")
print("=" * 80)

# Save final model
with open("payment_anomaly_model.pkl", "wb") as f:
    pickle.dump(payment_model, f)

# Save imputer
with open("payment_anomaly_imputer.pkl", "wb") as f:
    pickle.dump(imputer, f)

# Save feature list
with open("payment_anomaly_features.json", "w") as f:
    json.dump(candidate_features, f, indent=4)

# Save training-derived thresholds
payment_thresholds = {
    "p95": float(thresholds["p95"]),
    "p99": float(thresholds["p99"]),
    "p99.5": float(thresholds["p99.5"]),
    "p99.9": float(thresholds["p99.9"]),
    "primary_threshold": float(payment_threshold)
}

with open("payment_anomaly_thresholds.json", "w") as f:
    json.dump(payment_thresholds, f, indent=4)

# Save test results
export_cols = [
    "payment_id",
    "work_id",
    "vendor_name_normalized",
    "financial_year",
    "expenditure_date",
    "fund_disbursed_amount",
    "sanction_amount",
    "payment_sequence_number",
    "days_since_previous_payment",
    "days_since_sanction",
    "prior_disbursed_amount",
    "prior_vendor_count",
    "payment_share_of_prior_disbursed_amount",
    "vendor_previous_payment_count",
    "vendor_previous_work_count",
    "vendor_previous_total_amount",
    "anomaly_score",
    "payment_anomaly"
]

test_results[export_cols].to_csv(
    "payment_anomaly_results_2026_27.csv",
    index=False
)

print("\nExported:")
print("- payment_anomaly_model.pkl")
print("- payment_anomaly_imputer.pkl")
print("- payment_anomaly_features.json")
print("- payment_anomaly_thresholds.json")
print("- payment_anomaly_results_2026_27.csv")

print("\nPrimary threshold:", round(payment_threshold, 6))
print("Flagged payments:", int(test_results["payment_anomaly"].sum()))
print(
    "Flagged rate:",
    round(test_results["payment_anomaly"].mean() * 100, 4),
    "%"
)

print("\nPAYMENT ANOMALY MODEL EXPORT COMPLETE")

EXPORTING PAYMENT ANOMALY MODEL

Exported:
- payment_anomaly_model.pkl
- payment_anomaly_imputer.pkl
- payment_anomaly_features.json
- payment_anomaly_thresholds.json
- payment_anomaly_results_2026_27.csv

Primary threshold: 0.619199
Flagged payments: 83
Flagged rate: 0.9616 %

PAYMENT ANOMALY MODEL EXPORT COMPLETE


In [22]:
import pickle
import json
import numpy as np

print("=" * 80)
print("EXPORTED MODEL REPRODUCTION CHECK")
print("=" * 80)

# Reload exported artifacts
with open("payment_anomaly_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

with open("payment_anomaly_imputer.pkl", "rb") as f:
    loaded_imputer = pickle.load(f)

with open("payment_anomaly_features.json", "r") as f:
    loaded_features = json.load(f)

with open("payment_anomaly_thresholds.json", "r") as f:
    loaded_thresholds = json.load(f)

# Recreate test matrix
X_test_reload = test_df[loaded_features].copy()
X_test_reload = loaded_imputer.transform(X_test_reload)

# Recalculate scores
reproduced_scores = -loaded_model.score_samples(X_test_reload)

# Recalculate anomaly flags
reproduced_flags = (
    reproduced_scores >= loaded_thresholds["primary_threshold"]
)

# Compare with original results
original_scores = test_results["anomaly_score"].values
original_flags = test_results["payment_anomaly"].values

print("\nFeature list identical:",
      loaded_features == candidate_features)

print("Scores identical:",
      np.allclose(
          original_scores,
          reproduced_scores,
          rtol=1e-12,
          atol=1e-12
      ))

print("Flags identical:",
      np.array_equal(
          original_flags,
          reproduced_flags
      ))

print("\nOriginal flagged payments:",
      original_flags.sum())

print("Reproduced flagged payments:",
      reproduced_flags.sum())

print("\nMaximum score difference:",
      np.max(
          np.abs(original_scores - reproduced_scores)
      ))

if (
    loaded_features == candidate_features
    and np.allclose(original_scores, reproduced_scores)
    and np.array_equal(original_flags, reproduced_flags)
):
    print("\n" + "=" * 80)
    print("REPRODUCTION CHECK: PASS")
    print("PAYMENT ANOMALY MODEL: FINAL / FROZEN")
    print("=" * 80)
else:
    print("\nREPRODUCTION CHECK: FAILED")

EXPORTED MODEL REPRODUCTION CHECK

Feature list identical: True
Scores identical: True
Flags identical: True

Original flagged payments: 83
Reproduced flagged payments: 83

Maximum score difference: 0.0

REPRODUCTION CHECK: PASS
PAYMENT ANOMALY MODEL: FINAL / FROZEN


In [23]:
import json

print("=" * 80)
print("CREATING PAYMENT ANOMALY PROJECT ARTIFACTS")
print("=" * 80)

# --------------------------------------------------
# 1. Save train/test datasets
# --------------------------------------------------

train_df.to_csv(
    "payment_anomaly_train.csv",
    index=False
)

test_df.to_csv(
    "payment_anomaly_test.csv",
    index=False
)

# Final versions containing the model features
train_df[candidate_features].to_csv(
    "payment_anomaly_train_final.csv",
    index=False
)

test_results[
    candidate_features + ["anomaly_score", "payment_anomaly"]
].to_csv(
    "payment_anomaly_test_final.csv",
    index=False
)

# --------------------------------------------------
# 2. Save feature documentation
# --------------------------------------------------

feature_info = pd.DataFrame({
    "feature": candidate_features
})

feature_info.to_csv(
    "payment_anomaly_features.csv",
    index=False
)

# --------------------------------------------------
# 3. Save model metadata
# --------------------------------------------------

metadata = {
    "model_type": "IsolationForest",
    "anomaly_unit": "payment",
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "train_years": train_years,
    "test_years": test_years,
    "n_features": len(candidate_features),
    "n_estimators": 300,
    "random_state": 42,
    "primary_threshold_method": "training_p99",
    "primary_threshold": float(payment_threshold),
    "test_anomaly_count": int(test_results["payment_anomaly"].sum()),
    "test_anomaly_rate_percent": float(
        test_results["payment_anomaly"].mean() * 100
    ),
    "stable_anomalies_ge_4_of_5_seeds": int(
        (flag_frequency >= 4).sum()
    ),
    "amount_anomaly_score_correlation": 0.3001,
    "structural_missing_value_strategy": "constant_-1",
    "validation_status": "PASSED",
    "model_status": "FINAL / FROZEN"
}

with open(
    "payment_anomaly_metadata.json",
    "w"
) as f:
    json.dump(metadata, f, indent=4)

print("\nCreated:")
print("- payment_anomaly_train.csv")
print("- payment_anomaly_test.csv")
print("- payment_anomaly_train_final.csv")
print("- payment_anomaly_test_final.csv")
print("- payment_anomaly_features.csv")
print("- payment_anomaly_metadata.json")

print("\nPAYMENT ANOMALY ARTIFACT SET COMPLETE")

CREATING PAYMENT ANOMALY PROJECT ARTIFACTS

Created:
- payment_anomaly_train.csv
- payment_anomaly_test.csv
- payment_anomaly_train_final.csv
- payment_anomaly_test_final.csv
- payment_anomaly_features.csv
- payment_anomaly_metadata.json

PAYMENT ANOMALY ARTIFACT SET COMPLETE


In [24]:
print("Payment anomaly score range:")
print("Min:", round(test_results["anomaly_score"].min(), 4))
print("Max:", round(test_results["anomaly_score"].max(), 4))

print("\nPrimary threshold:")
print(round(payment_threshold, 6))

Payment anomaly score range:
Min: 0.3214
Max: 0.7593

Primary threshold:
0.619199
